<a href="https://colab.research.google.com/github/Minal-2007/AeroAssist-AI/blob/main/Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 28.8 MB/s eta 0:00:00


In [2]:
!streamlit --version

Streamlit, version 1.60.0


In [10]:
%%writefile app.py

import streamlit as st
import faiss
import pickle
import google.generativeai as genai

from sentence_transformers import SentenceTransformer

# -------------------------
# PAGE
# -------------------------

st.set_page_config(
    page_title="AeroAssist AI",
    page_icon="✈️",
    layout="wide"
)

st.title("✈️ AeroAssist AI")

st.caption("Aircraft Maintenance Assistant")

# -------------------------
# LOAD MODEL
# -------------------------

@st.cache_resource
def load_embedding_model():

    return SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

embedding_model = load_embedding_model()

# -------------------------
# LOAD VECTOR DB
# -------------------------

@st.cache_resource
def load_vector_db():

    index = faiss.read_index("aeroassist_index.faiss")

    with open("metadata.pkl","rb") as f:

        metadata = pickle.load(f)

    return index, metadata

index, metadata = load_vector_db()

# -------------------------
# GEMINI
# -------------------------

GOOGLE_API_KEY = "AQ.Ab8RN6KK4Zl4tvw6_JUQNVYB70J7Jr4ZctOQtpDQdHvFPPEmSg"

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-3.5-flash")

# -------------------------
# RETRIEVAL
# -------------------------

def retrieve(query, k=5):

    embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        embedding,
        k
    )

    docs = []

    for idx in indices[0]:

        docs.append(metadata[idx])

    return docs

# -------------------------
# PROMPT
# -------------------------

def build_prompt(query):

    docs = retrieve(query)

    context = ""

    for doc in docs:

        context += doc["text"]

        context += "\n\n"

    prompt = f"""
You are AeroAssist AI.

Answer ONLY from the FAA manuals below.

If the answer is not found,
say that the manuals do not contain sufficient information.

Context:

{context}

Question:

{query}

Answer:
"""

    return prompt

# -------------------------
# ASK
# -------------------------

question = st.text_input(
    "Ask an aviation maintenance question"
)

if st.button("Ask"):

    if question.strip() == "":

        st.warning("Enter a question.")

    else:

        with st.spinner("Searching FAA manuals..."):

            prompt = build_prompt(question)

            response = model.generate_content(prompt)

        st.success("Answer")

        st.write(response.text)

Overwriting app.py


In [4]:
!streamlit run app.py &> streamlit.log &

In [5]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 3s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [8]:
!cat tunnel.txt

your url is: https://old-worlds-care.loca.lt


In [6]:
!lt --port 8501 > tunnel.txt 2>&1 &

In [9]:
!curl https://loca.lt/mytunnelpassword

34.24.56.124

In [11]:
!pkill streamlit

In [12]:
!streamlit run app.py &> streamlit.log &

In [13]:
!pip install -q faiss-cpu sentence-transformers google-generativeai streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.1 MB/s eta 0:00:00


In [14]:
import faiss

print("FAISS Version:", faiss.__version__)

FAISS Version: 1.14.3


In [15]:
!pkill streamlit

In [16]:
!streamlit run app.py > streamlit.log 2>&1 &

In [17]:
!lt --port 8501 > tunnel.txt 2>&1 &

In [19]:
!cat tunnel.txt

your url is: https://forty-rooms-run.loca.lt


In [20]:
!curl https://loca.lt/mytunnelpassword

34.24.56.124